# Lab 4: Grover search

**Goal:** find a marked item among four in a single query.

Classically, with four unlabelled boxes and one prize, you need up to
three looks. Grover's algorithm uses interference to find it with one
query to the "oracle". For $N$ items it needs about $\sqrt{N}$ queries
instead of about $N$.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ms-cc-org/qc-workshop/blob/main/notebooks/04_grover_search.ipynb)

In [ ]:
# Install pinned versions (takes about a minute in Colab; already done locally)
%pip install -q qiskit==2.6.1 qiskit-aer==0.18.2 qiskit-ibm-runtime==0.51.0 matplotlib pylatexenc

In [ ]:
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from qiskit.visualization import plot_histogram

sim = AerSimulator()

def run(qc, shots=1000, backend=sim):
    """Run a circuit and return the measurement counts."""
    return backend.run(transpile(qc, backend), shots=shots).result().get_counts()

from qiskit.quantum_info import Statevector

## 4.1 The pieces

1. **Superposition:** H on both qubits gives equal amplitude to 00, 01,
   10 and 11.
2. **Oracle:** flips the sign of the marked state's amplitude. The
   probabilities don't change yet, so measuring now tells you nothing.
3. **Diffuser:** reflects every amplitude about the average. The marked
   state's amplitude grows and the others cancel.

Here is the oracle for the marked state `11`. CZ flips the sign of
$|11\rangle$ only.

In [ ]:
oracle_11 = QuantumCircuit(2, name="oracle")
oracle_11.cz(0, 1)

qc = QuantumCircuit(2)
qc.h([0, 1])
qc.compose(oracle_11, inplace=True)
print(Statevector(qc))  # the |11> amplitude is now negative

In [ ]:
def diffuser():
    qc = QuantumCircuit(2, name="diffuser")
    qc.h([0, 1])
    qc.x([0, 1])
    qc.cz(0, 1)
    qc.x([0, 1])
    qc.h([0, 1])
    return qc

def grover(oracle):
    qc = QuantumCircuit(2)
    qc.h([0, 1])
    qc.compose(oracle, inplace=True)
    qc.compose(diffuser(), inplace=True)
    qc.measure_all()
    return qc

qc = grover(oracle_11)
display(qc.draw("mpl"))
counts = run(qc)
print(counts)
plot_histogram(counts)

Every shot returns `11`: one query, certain answer.

## Exercise 4.1: Mark any state

Write `make_oracle(marked)` that flips the sign of any two-bit state,
for example `"01"`.

Hint: CZ only reacts to `11`. Apply X to each qubit that should be 0 in
the marked state, apply CZ, then undo the X gates. Remember that the
rightmost character is qubit 0.

In [ ]:
def make_oracle(marked):
    qc = QuantumCircuit(2, name=f"oracle {marked}")
    # TODO
    return qc

for marked in ["00", "01", "10", "11"]:
    counts = run(grover(make_oracle(marked)))
    print(marked, "->", counts)

## Exercise 4.2: Grover on noisy hardware

Run your search on the noisy simulator from Lab 3. Does the marked state
still win?

In [ ]:
from qiskit_ibm_runtime.fake_provider import FakeManilaV2
noisy = AerSimulator.from_backend(FakeManilaV2())

# TODO: run grover(make_oracle("01")) on `noisy` and plot the histogram

## Why this matters

With two qubits the speed-up is small. For a million items, Grover needs
about 1,000 queries where a classical search needs up to a million. The
same amplify-the-right-answer idea appears in many quantum algorithms.